# atari_reproducibility_deterministic_ops

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import return_curve, subsample
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load every replicate's stored runs per game, skipping replicates not stored yet.
SETTING = "dqn_atari"
REPLICATES = [
    component
    for component in EXPERIMENT.components
    if component.name.startswith(f"{SETTING}_replicate_")
]
GAMES = REPLICATES[0].sweep["ENV_HYPERS.GAME"]
FRAMESKIP = REPLICATES[0].config.ENV_HYPERS.FRAMESKIP

results = {}
for game in GAMES:
    stored = [
        load_results(EXPERIMENT, replicate.name, where={"ENV_HYPERS.GAME": game})
        for replicate in REPLICATES
    ]
    stored = [runs for runs in stored if runs is not None]
    if stored:
        results[game] = stored

In [ ]:
# Compute every replicate's return over frames per game, one row per replicate.
curves = {}
for game, stored in results.items():
    reward = np.concatenate([runs.reward for runs in stored])
    done = np.concatenate([runs.done for runs in stored])
    timesteps, returns = subsample(return_curve(reward, done))
    curves[game] = (timesteps * FRAMESKIP, returns)

In [ ]:
# Style for the per-game replicate plots.
STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
}
REPLICATE_LINE = {"linewidth": 1.5, "alpha": 0.5}
XLABEL = "Frames"
YLABEL = "Return"
YLABEL_KW = {"va": "center", "labelpad": 20}
XTICKS = np.array([1, 2, 3, 4]) * 100_000
XTICK_LABELS = [f"{tick // 1_000}k" for tick in XTICKS]


def title(game):
    return f"{game.replace('_', ' ').title()} (Deterministic)"

In [ ]:
# Plot each replicate as its own translucent curve per game, so overlaps show.
figures = {}
for game, (frames, returns) in curves.items():
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for replicate in returns:
            ax.plot(frames, replicate, **REPLICATE_LINE)
        ax.set(title=title(game), xlabel=XLABEL)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        fig.tight_layout()
    figures[f"{SETTING}_{game}"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF and a PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for extension in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{extension}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")